In [ ]:
!pip install -q ucimlrepo openml pandas numpy scikit-learn matplotlib seaborn joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from ucimlrepo import fetch_ucirepo
from sklearn.datasets import fetch_openml

import warnings
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid")

print("HEALTH INSIGHT - ML DEVELOPMENT")
print("Environment initialized successfully!")

# Fetch UCI Heart Disease Dataset
heart_dataset = fetch_ucirepo(id=45)

X_heart = heart_dataset.data.features.copy()
y_heart = heart_dataset.data.targets.copy()

# Prepare dataframe
heart_df = X_heart.copy()
heart_df["target"] = y_heart.iloc[:, 0]

# Convert target to numeric
heart_df["target"] = pd.to_numeric(
    heart_df["target"], errors="coerce"
)

# Binary classification:
# 0 = No heart disease
# 1 = Heart disease
heart_df["target"] = (
    heart_df["target"] > 0
).astype(int)

print("Heart Dataset Downloaded!")
print("Shape:", heart_df.shape)

display(heart_df.head())

heart_df.to_csv("heart.csv", index=False)

# Fetch Pima Indians Diabetes Dataset
diabetes_dataset = fetch_openml(
    data_id=37,
    as_frame=True
)

diabetes_df = diabetes_dataset.frame.copy()

# Standardize column names
diabetes_df.columns = [
    str(col).strip().lower() for col in diabetes_df.columns
]

# Separate target from features
target_col = diabetes_dataset.target_names[0]

diabetes_df = diabetes_df.rename(
    columns={str(target_col).lower(): "target"}
)

# Convert target into binary numerical values
diabetes_df["target"] = (
    diabetes_df["target"]
    .astype(str)
    .str.lower()
    .map({
        "tested_negative": 0,
        "tested_positive": 1
    })
)

print("Diabetes Dataset Downloaded!")
print("Shape:", diabetes_df.shape)

display(diabetes_df.head())

diabetes_df.to_csv("diabetes.csv", index=False)

print("HEART DISEASE DATASET")
print(heart_df.info())
print("\nMissing Values:")
print(heart_df.isnull().sum())

print("\nDIABETES DATASET")
print(diabetes_df.info())
print("\nMissing Values:")
print(diabetes_df.isnull().sum())

# Dataset dimensions
print("HEART DISEASE DATASET")
print("Shape:", heart_df.shape)
print("Features:", heart_df.columns.tolist())

print("\nDIABETES DATASET")
print("Shape:", diabetes_df.shape)
print("Features:", diabetes_df.columns.tolist())

# Statistical summaries
print("\nHeart Dataset Statistics:")
display(heart_df.describe().T)

print("\nDiabetes Dataset Statistics:")
display(diabetes_df.describe().T)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

heart_missing = heart_df.isnull().sum()
diabetes_missing = diabetes_df.isnull().sum()

sns.barplot(
    x=heart_missing.index,
    y=heart_missing.values,
    ax=axes[0]
)
axes[0].set_title("Heart Disease - Missing Values")
axes[0].tick_params(axis="x", rotation=90)

sns.barplot(
    x=diabetes_missing.index,
    y=diabetes_missing.values,
    ax=axes[1]
)
axes[1].set_title("Diabetes - Missing Values")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(
    data=heart_df,
    x="target",
    ax=axes[0]
)
axes[0].set_title("Heart Disease Class Distribution")
axes[0].set_xticks([0, 1], ["No Disease", "Disease"])

sns.countplot(
    data=diabetes_df,
    x="target",
    ax=axes[1]
)
axes[1].set_title("Diabetes Class Distribution")
axes[1].set_xticks([0, 1], ["No Diabetes", "Diabetes"])

plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

heart_corr = heart_df.corr(numeric_only=True)

sns.heatmap(
    heart_corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    ax=axes[0]
)
axes[0].set_title("Heart Disease Correlation Heatmap")

diabetes_corr = diabetes_df.corr(numeric_only=True)

sns.heatmap(
    diabetes_corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    ax=axes[1]
)
axes[1].set_title("Diabetes Correlation Heatmap")

plt.tight_layout()
plt.show()

# Heart disease feature distributions
heart_features = heart_df.select_dtypes(
    include=np.number
).drop(columns=["target"]).columns

heart_df[heart_features].hist(
    figsize=(16, 12),
    bins=20,
    edgecolor="black"
)

plt.suptitle(
    "Heart Disease - Feature Distributions",
    fontsize=16
)
plt.tight_layout()
plt.show()


# Diabetes feature distributions
diabetes_features = diabetes_df.select_dtypes(
    include=np.number
).drop(columns=["target"]).columns

diabetes_df[diabetes_features].hist(
    figsize=(16, 12),
    bins=20,
    edgecolor="black"
)

plt.suptitle(
    "Diabetes - Feature Distributions",
    fontsize=16
)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Heart dataset
sns.boxplot(
    data=heart_df,
    x="target",
    y="age",
    ax=axes[0, 0]
)
axes[0, 0].set_title("Heart Disease vs Age")

sns.boxplot(
    data=heart_df,
    x="target",
    y="chol",
    ax=axes[0, 1]
)
axes[0, 1].set_title("Heart Disease vs Cholesterol")

# Diabetes dataset
sns.boxplot(
    data=diabetes_df,
    x="target",
    y="plas",
    ax=axes[1, 0]
)
axes[1, 0].set_title("Diabetes vs Glucose (plas)")

sns.boxplot(
    data=diabetes_df,
    x="target",
    y="mass",
    ax=axes[1, 1]
)
axes[1, 1].set_title("Diabetes vs BMI (mass)")

plt.tight_layout()
plt.show()

zero_columns = [
    col for col in ["plas", "pres", "skin", "insu", "mass",
                    "glucose", "bloodpressure", "skinthickness",
                    "insulin", "bmi"]
    if col in diabetes_df.columns
]

print("Zero values in diabetes measurements:\n")

for col in zero_columns:
    count = (diabetes_df[col] == 0).sum()
    print(f"{col}: {count}")

print("Heart Dataset Duplicates:",
      heart_df.duplicated().sum())

print("Diabetes Dataset Duplicates:",
      diabetes_df.duplicated().sum())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    data=heart_df.drop(columns=["target"]),
    ax=axes[0]
)
axes[0].set_title("Heart Dataset - Outliers")
axes[0].tick_params(axis="x", rotation=90)

sns.boxplot(
    data=diabetes_df.drop(columns=["target"]),
    ax=axes[1]
)
axes[1].set_title("Diabetes Dataset - Outliers")
axes[1].tick_params(axis="x", rotation=90)

plt.tight_layout()
plt.show()

# Create independent copies
heart_clean = heart_df.copy()
diabetes_clean = diabetes_df.copy()

# Convert all feature columns to numeric where possible
for col in heart_clean.columns:
    heart_clean[col] = pd.to_numeric(
        heart_clean[col], errors="coerce"
    )

for col in diabetes_clean.columns:
    diabetes_clean[col] = pd.to_numeric(
        diabetes_clean[col], errors="coerce"
    )

# Remove rows with missing target values
heart_clean = heart_clean.dropna(subset=["target"])
diabetes_clean = diabetes_clean.dropna(subset=["target"])

# Remove exact duplicate records
heart_clean = heart_clean.drop_duplicates().reset_index(drop=True)
diabetes_clean = diabetes_clean.drop_duplicates().reset_index(drop=True)

# Ensure binary targets
heart_clean["target"] = heart_clean["target"].astype(int)
diabetes_clean["target"] = diabetes_clean["target"].astype(int)

print("HEART DATASET:", heart_clean.shape)
print("DIABETES DATASET:", diabetes_clean.shape)

print("\nHeart target distribution:")
print(heart_clean["target"].value_counts())

print("\nDiabetes target distribution:")
print(diabetes_clean["target"].value_counts())

# Columns where zero represents an invalid/missing measurement
possible_zero_columns = [
    "plas", "pres", "skin", "insu", "mass",
    "glucose", "bloodpressure", "skinthickness",
    "insulin", "bmi"
]

zero_columns = [
    col for col in possible_zero_columns
    if col in diabetes_clean.columns
]

# Replace invalid zeros with NaN
diabetes_clean[zero_columns] = (
    diabetes_clean[zero_columns].replace(0, np.nan)
)

print("Zero values replaced with missing values.")

print("\nMissing values after zero correction:")
display(diabetes_clean.isnull().sum().to_frame("Missing Values"))

# Heart disease categorical features
heart_categorical = [
    "sex", "cp", "fbs", "restecg",
    "exang", "slope", "ca", "thal"
]

heart_categorical = [
    col for col in heart_categorical
    if col in heart_clean.columns
]

heart_numerical = [
    col for col in heart_clean.columns
    if col not in heart_categorical + ["target"]
]

# Diabetes features
diabetes_numerical = [
    col for col in diabetes_clean.columns
    if col != "target"
]

print("HEART DISEASE FEATURES")
print("Categorical:", heart_categorical)
print("Numerical:", heart_numerical)

print("\nDIABETES FEATURES")
print("Numerical:", diabetes_numerical)

# Heart Disease
X_heart = heart_clean.drop(columns=["target"])
y_heart = heart_clean["target"]

# Diabetes
X_diabetes = diabetes_clean.drop(columns=["target"])
y_diabetes = diabetes_clean["target"]

print("Heart X:", X_heart.shape)
print("Heart y:", y_heart.shape)

print("\nDiabetes X:", X_diabetes.shape)
print("Diabetes y:", y_diabetes.shape)

from sklearn.model_selection import train_test_split

# Heart Disease
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    X_heart,
    y_heart,
    test_size=0.20,
    random_state=42,
    stratify=y_heart
)

# Diabetes
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_diabetes,
    y_diabetes,
    test_size=0.20,
    random_state=42,
    stratify=y_diabetes
)

print("HEART DATASET")
print("Training:", Xh_train.shape)
print("Testing:", Xh_test.shape)

print("\nDIABETES DATASET")
print("Training:", Xd_train.shape)
print("Testing:", Xd_test.shape)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Heart Disease preprocessing
heart_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, heart_numerical),
    ("categorical", categorical_pipeline, heart_categorical)
])

# Diabetes preprocessing
diabetes_preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, diabetes_numerical)
])

print("Heart preprocessing pipeline created!")
print("Diabetes preprocessing pipeline created!")

# Fit preprocessing ONLY on training data
Xh_train_processed = heart_preprocessor.fit_transform(Xh_train)
Xh_test_processed = heart_preprocessor.transform(Xh_test)

Xd_train_processed = diabetes_preprocessor.fit_transform(Xd_train)
Xd_test_processed = diabetes_preprocessor.transform(Xd_test)

# Validation
def validate_data(name, train_data, test_data):
    train_array = train_data.toarray() if hasattr(
        train_data, "toarray"
    ) else np.asarray(train_data)

    test_array = test_data.toarray() if hasattr(
        test_data, "toarray"
    ) else np.asarray(test_data)

    print(f"\n{name}")
    print("Training shape:", train_array.shape)
    print("Testing shape:", test_array.shape)
    print("Training finite:", np.isfinite(train_array).all())
    print("Testing finite:", np.isfinite(test_array).all())

validate_data(
    "HEART DISEASE",
    Xh_train_processed,
    Xh_test_processed
)

validate_data(
    "DIABETES",
    Xd_train_processed,
    Xd_test_processed
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

from sklearn.base import clone
from sklearn.pipeline import Pipeline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("Machine Learning environment ready!")

def create_models():
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        ),

        "Random Forest": RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_split=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1
        ),

        "SVM": SVC(
            kernel="rbf",
            C=1.0,
            probability=True,
            class_weight="balanced",
            random_state=42
        ),

        "Decision Tree": DecisionTreeClassifier(
            class_weight="balanced",
            random_state=42
        ),

        "KNN": KNeighborsClassifier(
            n_neighbors=5,
            weights="distance"
        )
    }


heart_models = create_models()
diabetes_models = create_models()

print("5 Heart Disease models initialized.")
print("5 Diabetes models initialized.")

def train_and_evaluate(
    models,
    preprocessor,
    X_train,
    X_test,
    y_train,
    y_test,
    disease_name
):

    results = []
    trained_models = {}

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for model_name, classifier in models.items():

        print(f"\nTraining {disease_name}: {model_name}")

        # Complete preprocessing + model pipeline
        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(classifier))
        ])

        # Cross-validation on training data only
        cv_scores = cross_val_score(
            pipeline,
            X_train,
            y_train,
            cv=cv,
            scoring="f1",
            n_jobs=-1
        )

        # Train on complete training split
        pipeline.fit(X_train, y_train)

        # Predictions on test split
        y_pred = pipeline.predict(X_test)

        # Probability scores for ROC-AUC
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        # Evaluation metrics
        metrics = {
            "Disease": disease_name,
            "Model": model_name,
            "CV F1 Mean": cv_scores.mean(),
            "CV F1 Std": cv_scores.std(),
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(
                y_test, y_pred, zero_division=0
            ),
            "Recall": recall_score(
                y_test, y_pred, zero_division=0
            ),
            "F1 Score": f1_score(
                y_test, y_pred, zero_division=0
            ),
            "ROC AUC": roc_auc_score(y_test, y_prob)
        }

        results.append(metrics)
        trained_models[model_name] = pipeline

        print(
            f"Accuracy: {metrics['Accuracy']:.4f} | "
            f"F1: {metrics['F1 Score']:.4f} | "
            f"ROC-AUC: {metrics['ROC AUC']:.4f}"
        )

    results_df = pd.DataFrame(results)

    return results_df, trained_models

heart_results, heart_trained_models = train_and_evaluate(
    models=heart_models,
    preprocessor=heart_preprocessor,
    X_train=Xh_train,
    X_test=Xh_test,
    y_train=yh_train,
    y_test=yh_test,
    disease_name="Heart Disease"
)

print("\nHEART DISEASE MODEL COMPARISON")

display(
    heart_results.sort_values(
        by="F1 Score",
        ascending=False
    ).round(4)
)

diabetes_results, diabetes_trained_models = train_and_evaluate(
    models=diabetes_models,
    preprocessor=diabetes_preprocessor,
    X_train=Xd_train,
    X_test=Xd_test,
    y_train=yd_train,
    y_test=yd_test,
    disease_name="Diabetes"
)

print("\nDIABETES MODEL COMPARISON")

display(
    diabetes_results.sort_values(
        by="F1 Score",
        ascending=False
    ).round(4)
)

all_results = pd.concat(
    [heart_results, diabetes_results],
    ignore_index=True
)

print("HEALTH INSIGHT - COMPLETE MODEL COMPARISON")

display(
    all_results.sort_values(
        by=["Disease", "F1 Score"],
        ascending=[True, False]
    ).round(4)
)

metrics_to_plot = [
    "Accuracy",
    "Precision",
    "Recall",
    "F1 Score",
    "ROC AUC"
]

for disease in ["Heart Disease", "Diabetes"]:

    disease_results = all_results[
        all_results["Disease"] == disease
    ]

    plt.figure(figsize=(12, 6))

    disease_results.set_index("Model")[
        metrics_to_plot
    ].plot(
        kind="bar",
        figsize=(12, 6),
        ylim=(0, 1.05),
        width=0.8
    )

    plt.title(f"{disease} - ML Model Performance Comparison")
    plt.ylabel("Performance Score")
    plt.xlabel("Machine Learning Algorithm")
    plt.xticks(rotation=25, ha="right")
    plt.legend(loc="lower right")
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_confusion_matrices(
    models,
    X_test,
    y_test,
    disease_name
):

    fig, axes = plt.subplots(
        2, 3,
        figsize=(15, 9)
    )

    axes = axes.flatten()

    for i, (name, model) in enumerate(models.items()):

        predictions = model.predict(X_test)

        cm = confusion_matrix(
            y_test,
            predictions,
            labels=[0, 1]
        )

        sns.heatmap(
            cm,
            annot=True,
            fmt="d",
            cmap="Blues",
            xticklabels=["Negative", "Positive"],
            yticklabels=["Negative", "Positive"],
            ax=axes[i]
        )

        axes[i].set_title(name)
        axes[i].set_xlabel("Predicted")
        axes[i].set_ylabel("Actual")

    axes[5].axis("off")

    plt.suptitle(
        f"{disease_name} - Confusion Matrices",
        fontsize=16
    )

    plt.tight_layout()
    plt.show()


plot_confusion_matrices(
    heart_trained_models,
    Xh_test,
    yh_test,
    "Heart Disease"
)

plot_confusion_matrices(
    diabetes_trained_models,
    Xd_test,
    yd_test,
    "Diabetes"
)

def plot_roc_curves(
    models,
    X_test,
    y_test,
    disease_name
):

    plt.figure(figsize=(9, 7))

    for name, model in models.items():

        RocCurveDisplay.from_estimator(
            model,
            X_test,
            y_test,
            name=name,
            ax=plt.gca()
        )

    plt.plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        label="Random Classifier"
    )

    plt.title(f"{disease_name} - ROC Curve Comparison")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


plot_roc_curves(
    heart_trained_models,
    Xh_test,
    yh_test,
    "Heart Disease"
)

plot_roc_curves(
    diabetes_trained_models,
    Xd_test,
    yd_test,
    "Diabetes"
)

for disease, models, X_test, y_test in [
    ("HEART DISEASE", heart_trained_models, Xh_test, yh_test),
    ("DIABETES", diabetes_trained_models, Xd_test, yd_test)
]:

    print("\n" + "=" * 60)
    print(disease)
    print("=" * 60)

    for name, model in models.items():

        predictions = model.predict(X_test)

        print(f"\nMODEL: {name}")

        print(
            classification_report(
                y_test,
                predictions,
                target_names=["Negative", "Positive"],
                zero_division=0
            )
        )

heart_param_grids = {
    "Logistic Regression": {
        "classifier__C": [0.01, 0.1, 1, 10, 100],
        "classifier__solver": ["liblinear", "lbfgs"]
    },

    "Random Forest": {
        "classifier__n_estimators": [100, 200, 300],
        "classifier__max_depth": [None, 5, 10, 20],
        "classifier__min_samples_split": [2, 5, 10]
    },

    "SVM": {
        "classifier__C": [0.1, 1, 10],
        "classifier__kernel": ["linear", "rbf"],
        "classifier__gamma": ["scale", "auto"]
    },

    "Decision Tree": {
        "classifier__max_depth": [None, 3, 5, 10, 15],
        "classifier__min_samples_split": [2, 5, 10],
        "classifier__criterion": ["gini", "entropy"]
    },

    "KNN": {
        "classifier__n_neighbors": [3, 5, 7, 9, 11],
        "classifier__weights": ["uniform", "distance"],
        "classifier__metric": ["euclidean", "manhattan"]
    }
}

# Use the same search spaces for both datasets
diabetes_param_grids = heart_param_grids.copy()

print("Hyperparameter search spaces configured!")

from sklearn.model_selection import GridSearchCV

def tune_models(
    models,
    param_grids,
    preprocessor,
    X_train,
    y_train,
    disease_name
):

    tuned_models = {}
    tuning_results = []

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42
    )

    for model_name, classifier in models.items():

        print("\n" + "=" * 55)
        print(f"TUNING: {disease_name} - {model_name}")
        print("=" * 55)

        pipeline = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("classifier", clone(classifier))
        ])

        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grids[model_name],
            scoring="f1",
            cv=cv,
            n_jobs=-1,
            refit=True,
            verbose=0,
            error_score="raise"
        )

        grid.fit(X_train, y_train)

        tuned_models[model_name] = grid.best_estimator_

        tuning_results.append({
            "Disease": disease_name,
            "Model": model_name,
            "Best CV F1": grid.best_score_,
            "Best Parameters": grid.best_params_
        })

        print("Best CV F1:", round(grid.best_score_, 4))
        print("Best Parameters:", grid.best_params_)

    return tuned_models, pd.DataFrame(tuning_results)

heart_tuned_models, heart_tuning_results = tune_models(
    models=heart_models,
    param_grids=heart_param_grids,
    preprocessor=heart_preprocessor,
    X_train=Xh_train,
    y_train=yh_train,
    disease_name="Heart Disease"
)

display(
    heart_tuning_results.sort_values(
        by="Best CV F1",
        ascending=False
    )
)

diabetes_tuned_models, diabetes_tuning_results = tune_models(
    models=diabetes_models,
    param_grids=diabetes_param_grids,
    preprocessor=diabetes_preprocessor,
    X_train=Xd_train,
    y_train=yd_train,
    disease_name="Diabetes"
)

display(
    diabetes_tuning_results.sort_values(
        by="Best CV F1",
        ascending=False
    )
)

tuning_results = pd.concat(
    [
        heart_tuning_results,
        diabetes_tuning_results
    ],
    ignore_index=True
)

print("HEALTH INSIGHT - HYPERPARAMETER TUNING RESULTS")

display(
    tuning_results.sort_values(
        by=["Disease", "Best CV F1"],
        ascending=[True, False]
    ).round(4)
)

plt.figure(figsize=(14, 6))

sns.barplot(
    data=tuning_results,
    x="Model",
    y="Best CV F1",
    hue="Disease"
)

plt.title("Optimized ML Models - Cross-Validation F1 Comparison")
plt.xlabel("Machine Learning Algorithm")
plt.ylabel("Best Cross-Validation F1 Score")
plt.ylim(0, 1)
plt.xticks(rotation=20)
plt.legend(title="Disease")
plt.tight_layout()
plt.show()

# Select best Heart Disease model
best_heart_row = heart_tuning_results.loc[
    heart_tuning_results["Best CV F1"].idxmax()
]

best_heart_name = best_heart_row["Model"]

final_heart_model = heart_tuned_models[
    best_heart_name
]


# Select best Diabetes model
best_diabetes_row = diabetes_tuning_results.loc[
    diabetes_tuning_results["Best CV F1"].idxmax()
]

best_diabetes_name = best_diabetes_row["Model"]

final_diabetes_model = diabetes_tuned_models[
    best_diabetes_name
]


print("FINAL HEART DISEASE MODEL:", best_heart_name)
print(
    "Cross-validation F1:",
    round(best_heart_row["Best CV F1"], 4)
)

print("\nFINAL DIABETES MODEL:", best_diabetes_name)
print(
    "Cross-validation F1:",
    round(best_diabetes_row["Best CV F1"], 4)
)

def final_evaluation(model, X_test, y_test, disease_name):

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    results = {
        "Disease": disease_name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1 Score": f1_score(
            y_test, predictions, zero_division=0
        ),
        "ROC AUC": roc_auc_score(y_test, probabilities)
    }

    print("\n", "=" * 50)
    print(disease_name, "- FINAL MODEL")
    print("=" * 50)

    print("Accuracy:", round(results["Accuracy"], 4))
    print("Precision:", round(results["Precision"], 4))
    print("Recall:", round(results["Recall"], 4))
    print("F1 Score:", round(results["F1 Score"], 4))
    print("ROC AUC:", round(results["ROC AUC"], 4))

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            target_names=["Negative", "Positive"],
            zero_division=0
        )
    )

    return results


final_heart_metrics = final_evaluation(
    final_heart_model,
    Xh_test,
    yh_test,
    "Heart Disease"
)

final_diabetes_metrics = final_evaluation(
    final_diabetes_model,
    Xd_test,
    yd_test,
    "Diabetes"
)

final_metrics_df = pd.DataFrame([
    final_heart_metrics,
    final_diabetes_metrics
])

display(final_metrics_df.round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

final_models_info = [
    ("Heart Disease", final_heart_model, Xh_test, yh_test),
    ("Diabetes", final_diabetes_model, Xd_test, yd_test)
]

for i, (name, model, X_test, y_test) in enumerate(final_models_info):

    predictions = model.predict(X_test)

    cm = confusion_matrix(
        y_test,
        predictions,
        labels=[0, 1]
    )

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"],
        ax=axes[i, 0]
    )

    axes[i, 0].set_title(name + " - Confusion Matrix")
    axes[i, 0].set_xlabel("Predicted")
    axes[i, 0].set_ylabel("Actual")

    RocCurveDisplay.from_estimator(
        model,
        X_test,
        y_test,
        ax=axes[i, 1]
    )

    axes[i, 1].set_title(name + " - ROC Curve")

plt.tight_layout()
plt.show()

import os
import joblib

os.makedirs("health_insight_models", exist_ok=True)

# Save complete pipelines
joblib.dump(
    final_heart_model,
    "health_insight_models/heart_model.pkl"
)

joblib.dump(
    final_diabetes_model,
    "health_insight_models/diabetes_model.pkl"
)

print("Both final models exported successfully!")

import json

with open(
    "health_insight_models/model_metrics.json",
    "w"
) as f:

    json.dump(
        {
            "heart_disease": final_heart_metrics,
            "diabetes": final_diabetes_metrics
        },
        f,
        indent=4
    )

print("Model metrics saved!")

metadata = {
    "project": "HEALTH INSIGHT",
    "heart_disease": {
        "model": best_heart_name,
        "features": list(X_heart.columns)
    },
    "diabetes": {
        "model": best_diabetes_name,
        "features": list(X_diabetes.columns)
    }
}

with open(
    "health_insight_models/model_metadata.json",
    "w"
) as f:

    json.dump(metadata, f, indent=4)

print("Model metadata saved!")

!zip -r HEALTH_INSIGHT_MODELS.zip health_insight_models

from google.colab import files

files.download("HEALTH_INSIGHT_MODELS.zip")